# Ballistics gate diagnostic (post hoc; does not rescue the pilot)

Why did the calibration gate fail (Spearman -0.08)? Recomputes it from saved predictions:
1. all 28 anchor pairs (reproduces the failure);
2. excluding the collapsed cheap DINOv2 model (21 pairs);
3. strength-adjusted: severe co-miss residualised on the pair's mean and gap in severe misses, cheap vs full;
4. whether cheap training preserved each anchor's strength ranking (severe misses, cheap vs full).

CPU, under 2 minutes. Writes `04_results/ballistics/gate_diagnostic.json`.

In [2]:
from google.colab import drive
drive.mount('/content/drive')
import json, itertools, numpy as np, pandas as pd
from scipy.stats import spearmanr, rankdata
BASE = '/content/drive/MyDrive/stochastic_orthogonality/data/dr_ensemble_v2'; ZOO_OUT = f'{BASE}/04_results/ballistics'
ANCHOR_FULL = {'resnet50': 'resnet50_5class', 'resnet50_swsl': 'resnet50_swsl_5class', 'densenet121': 'densenet121_5class', 'efficientnet_b3': 'efficientnet_b3_5class',
               'vit_b': 'vit_base_5class', 'vit_b_mae': 'vit_base_mae_5class', 'vit_b_dinov2': 'vit_base_dinov2_5class', 'retfound': 'retfound_5class'}
fr = pd.read_csv(f'{BASE}/01_splits/eyepacs_split_fresh_v2.csv', dtype={'image_id': str}).sort_values('image_id').reset_index(drop=True)
IDS = fr.image_id.to_numpy(); G = fr[next(c for c in ('level','grade') if c in fr.columns)].to_numpy().astype(int); sev = G >= 3
def load(path):
    df = pd.read_csv(path, dtype={'image_id': str}); pc = next(c for c in ('predicted_probability','probability','prob') if c in df.columns)
    return df.drop_duplicates('image_id').set_index('image_id')[pc].astype(float).reindex(IDS).to_numpy()
def refer(s, f=0.25):
    r = np.zeros(len(s), bool); r[np.argsort(-s, kind='stable')[:int(round(f * len(s)))]] = True; return r
cheap = {a: refer(load(f'{ZOO_OUT}/preds/{a}_fresh.csv')) for a in ANCHOR_FULL}
full = {a: refer(load(f'{BASE}/03_predictions/eyepacs_predictions_fresh_{v}_v2.csv')) for a, v in ANCHOR_FULL.items()}
def comiss(ra, rb):
    a, b = sev & ~ra, sev & ~rb; T = sev.sum(); return ((a & b).sum() / T) / ((a.sum() / T) * (b.sum() / T))
miss = lambda R, a: int((sev & ~R[a]).sum())
def table(anchors):
    rows = []
    for a, b in itertools.combinations(anchors, 2):
        rows.append({'a': a, 'b': b, 'cheap': comiss(cheap[a], cheap[b]), 'full': comiss(full[a], full[b]),
                     'cm': (miss(cheap, a) + miss(cheap, b)) / 2, 'cg': abs(miss(cheap, a) - miss(cheap, b)),
                     'fm': (miss(full, a) + miss(full, b)) / 2, 'fg': abs(miss(full, a) - miss(full, b))})
    return pd.DataFrame(rows)
def resid(y, X):
    X = np.column_stack([np.ones(len(y)), X]); return y - X @ np.linalg.lstsq(X, y, rcond=None)[0]
OUT = {}
for label, anchors in (('all_28_pairs', list(ANCHOR_FULL)), ('without_dinov2_21_pairs', [a for a in ANCHOR_FULL if a != 'vit_b_dinov2'])):
    T = table(anchors)
    OUT[label] = {'n_pairs': len(T), 'spearman_raw': round(float(spearmanr(T.cheap, T.full)[0]), 3),
                  'spearman_strength_adjusted': round(float(spearmanr(resid(rankdata(T.cheap.values), rankdata(T[['cm','cg']].values, axis=0)),
                                                                      resid(rankdata(T.full.values), rankdata(T[['fm','fg']].values, axis=0)))[0]), 3)}
OUT['strength_ranking_preserved'] = {'spearman_severe_misses_cheap_vs_full': round(float(spearmanr([miss(cheap, a) for a in ANCHOR_FULL], [miss(full, a) for a in ANCHOR_FULL])[0]), 3),
                                    'severe_misses': {a: {'cheap': miss(cheap, a), 'full': miss(full, a)} for a in ANCHOR_FULL}}
json.dump(OUT, open(f'{ZOO_OUT}/gate_diagnostic.json', 'w'), indent=2); print(json.dumps(OUT, indent=1))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
{
 "all_28_pairs": {
  "n_pairs": 28,
  "spearman_raw": -0.081,
  "spearman_strength_adjusted": 0.437
 },
 "without_dinov2_21_pairs": {
  "n_pairs": 21,
  "spearman_raw": 0.637,
  "spearman_strength_adjusted": 0.66
 },
 "strength_ranking_preserved": {
  "spearman_severe_misses_cheap_vs_full": 0.287,
  "severe_misses": {
   "resnet50": {
    "cheap": 78,
    "full": 55
   },
   "resnet50_swsl": {
    "cheap": 25,
    "full": 28
   },
   "densenet121": {
    "cheap": 38,
    "full": 45
   },
   "efficientnet_b3": {
    "cheap": 49,
    "full": 45
   },
   "vit_b": {
    "cheap": 32,
    "full": 39
   },
   "vit_b_mae": {
    "cheap": 48,
    "full": 49
   },
   "vit_b_dinov2": {
    "cheap": 419,
    "full": 21
   },
   "retfound": {
    "cheap": 30,
    "full": 31
   }
  }
 }
}
